# 11 MQA and GQA

## Problem

MHA、MQA、GQA 到底分别在做什么折中？为什么很多现代模型既不选最原始的 MHA，也不直接极端地用 MQA，而是选择 GQA？

## Dependencies

建议先完成 `05_multi_head_attention.ipynb` 和 `10_kv_cache_and_inference.ipynb`。


## Goals

- 理解 MHA、MQA、GQA 的 Q/K/V 组织方式差异
- 理解 K/V head 数量为什么会直接影响 KV cache 成本
- 理解 GQA 为什么是表达能力和推理效率之间的工程折中
- 能根据 shape 说清三者的差异

## Scope and Approach

这一节不会只给三个定义。我们会从 KV cache 的成本问题出发，说明为什么“query 有多少个头”和“key/value 有多少个头”可以不是一回事，然后再看 MHA、MQA、GQA 各自怎么组织这些 head。


## 先回到问题本身：哪里贵

MHA 的问题不是它效果差，而是它在推理时比较贵。特别是当：

- head 数量多
- 上下文长
- 层数深

历史 K/V cache 的体积会迅速变大。于是工程上自然会问：

**是不是每个 query head 都必须拥有一套独立的 K/V head？**

如果答案是否定的，那 cache 体积就有机会降下来。


## 三种结构先一句话记住

- **MHA**：每个 query head 都配一套独立的 K/V head
- **MQA**：很多 query head 共享同一套 K/V
- **GQA**：若干个 query head 共享一套 K/V，也就是分组共享

所以真正变化的重点不是 Q，而是：

**K/V 到底共享到什么程度。**


In [ ]:
import numpy as np

hidden_dim = 16
num_query_heads = 8
head_dim = 4
seq_len = 32

def kv_cache_elements(seq_len, kv_heads, head_dim):
    # 乘 2 是因为 K 和 V 都要存。
    return seq_len * kv_heads * head_dim * 2

mha_kv_heads = 8
mqa_kv_heads = 1
gqa_kv_heads = 2

print('MHA cache elements =', kv_cache_elements(seq_len, mha_kv_heads, head_dim))
print('MQA cache elements =', kv_cache_elements(seq_len, mqa_kv_heads, head_dim))
print('GQA cache elements =', kv_cache_elements(seq_len, gqa_kv_heads, head_dim))


## MHA：表达最完整，但 K/V 最贵

在标准多头注意力里：

- 有多少个 query head
- 就有多少个 key head
- 也有多少个 value head

这意味着每个 head 都有最大自由度，但也意味着每个 head 的历史 K/V 都必须单独缓存。


In [ ]:
batch = 1
seq_len = 5

# MHA: q_heads = 8, kv_heads = 8
Q_mha = np.random.randn(batch, seq_len, 8, head_dim)
K_mha = np.random.randn(batch, seq_len, 8, head_dim)
V_mha = np.random.randn(batch, seq_len, 8, head_dim)

print('MHA shapes:')
print('Q_mha.shape =', Q_mha.shape)
print('K_mha.shape =', K_mha.shape)
print('V_mha.shape =', V_mha.shape)


## MQA：把 K/V 共享到极致

MQA 的思路非常激进：

- query 头还是很多
- 但所有 query head 共用同一套 K/V

这样做的直接好处是：

- cache 立刻大幅缩小
- 推理时 K/V 搬运成本也下降

但代价也很明显：所有 query head 都在看同一套 K/V 表示，灵活性会下降。


In [ ]:
# MQA: q_heads = 8, kv_heads = 1
Q_mqa = np.random.randn(batch, seq_len, 8, head_dim)
K_mqa = np.random.randn(batch, seq_len, 1, head_dim)
V_mqa = np.random.randn(batch, seq_len, 1, head_dim)

print('MQA shapes:')
print('Q_mqa.shape =', Q_mqa.shape)
print('K_mqa.shape =', K_mqa.shape)
print('V_mqa.shape =', V_mqa.shape)


## GQA：为什么它是现实里的折中方案

GQA 的思路比 MQA 更温和：

- 不是所有 query head 都共享一套 K/V
- 而是每几个 query head 组成一组，共享一套 K/V

这就形成了一个非常自然的折中：

- 比 MHA 省 cache
- 比 MQA 更保留多头表达自由度

所以很多实际系统会把它当成比较现实的平衡点。


In [ ]:
# GQA: q_heads = 8, kv_heads = 2
# 这里可以理解为：每 4 个 query head 共用一组 K/V。
Q_gqa = np.random.randn(batch, seq_len, 8, head_dim)
K_gqa = np.random.randn(batch, seq_len, 2, head_dim)
V_gqa = np.random.randn(batch, seq_len, 2, head_dim)

print('GQA shapes:')
print('Q_gqa.shape =', Q_gqa.shape)
print('K_gqa.shape =', K_gqa.shape)
print('V_gqa.shape =', V_gqa.shape)

# 这里明确写出 query head 到 kv group 的映射关系。
query_to_group = {q_head: q_head // 4 for q_head in range(8)}
print('GQA query_to_group =', query_to_group)


## 从 shape 看三者本质差别

如果只记一条最重要的判断标准，可以记这个：

- `Q` 的 head 数决定“有多少个查询视角”
- `K/V` 的 head 数决定“历史状态要存多少份表示”

所以：

- MHA：查询视角多，历史表示也存很多份
- MQA：查询视角多，但历史表示只存一份
- GQA：查询视角多，历史表示存少量分组份数

这就是为什么它们的核心 trade-off 会直接落到 KV cache 上。

## Common mistakes

- 把 MQA 理解成“少几个头”，而不是“query head 仍然很多，但 K/V 共享”。
- 只记定义，不理解为什么 K/V head 数量会直接影响 cache 成本。
- 以为 GQA 只是一个理论花活。它之所以常见，恰恰因为折中合理。
- 把 query head 数量和 kv head 数量默认视为必须相同。

## Checkpoints

- 假设 `num_query_heads = 16`，自己设计一个 `kv_heads = 4` 的 GQA 分组方案。
- 用自己的话解释为什么 MQA cache 最省，但不一定总能保留最灵活的表达能力。
- 说明 GQA 为什么会成为现实中的常见折中。
- 回答：三种结构里，真正变化最大的到底是 Q 还是 K/V？

## Summary

理解了 MHA、MQA、GQA，你就抓住了现代注意力工程优化的一条主线：query head 可以很多，但历史 K/V 不一定非得按同样份数完整存储。GQA 的价值就在于，它往往在表达能力和推理成本之间给出一个现实可用的折中点。

## Next Module

下一节进入 MLA。和 GQA 相比，MLA 会把“怎么省 cache”再往前推进一步：它不仅减少 head 份数，还会直接改变被缓存的状态表示形式。
